In [3]:
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import os
from importlib import reload
import time
import torch as T
from torchvision.transforms import GaussianBlur
import sys
sys.path.append('../scripts')
import landcover_definitions as lc
import util
from general_prior_calculation import calculate_prior_preblurred

In [2]:
pip install rasterio

  Using cached affine-3.0.1-py3-none-any.whl.metadata (4.5 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.1/39.1 MB 136.3 MB/s  0:00:00 eta 0:00:01
Using cached affine-3.0.1-py3-none-any.whl (10 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [rasterio]1/2 [rasterio]
Note: you may need to restart the kernel to use updated packages.


In [4]:
!pip install rasterio
!pip install torchvision

In [ ]:
# Step 1: data - done (Katherine's pairs)
# Step 2: hand coded cooccurrence matrix - done (simple based on meeting 10/23/2026)
# Step 3: create priors: one hot, blur, times matrix - NEED TO DO
# Step 4: use NAIP in CNN and prior

In [4]:
cyverse = True
if cyverse:
    use_gpu = True
    pairs_path = '/data-store/iplant/home/emilynagamoto/NAIP_LCMAP/'
else: 
    use_gpu = False
    pairs_path = 'C:/Users/emna9376/OneDrive - UCB-O365/CIRES_IRP/naip_lcmap_pairs/'

In [5]:
if use_gpu:
    device = T.device('cuda:0' if T.cuda.is_available() else 'cpu')
else:
    device = T.device('cpu')

In [6]:
########## STEP 1
lcmap_naip_dict = {'hayman':'hayman_co_m_3910554_se_13_060_20190913_20191121'}

# load one pair
file1_firename = 'hayman'
lcmap_file1 = pairs_path +  'hayman_lcmap.tif'
naip_file1 = pairs_path + 'hayman_co_m_3910554_se_13_060_20190913_20191121_image.tif'

In [5]:
# get the color table + what colors are in this file 
with rasterio.open(lcmap_file1) as src:
    arr = src.read(1)
    print(np.unique(arr))          # which codes actually appear in your tiles
    print(src.colormap(1))         # embedded color table, if the file has one
    print(src.tags(), src.descriptions)


[1 3 4 5 6 8]
{0: (0, 0, 0, 255), 1: (255, 50, 50, 255), 2: (190, 140, 90, 255), 3: (230, 240, 210, 255), 4: (28, 99, 48, 255), 5: (0, 112, 255, 255), 6: (179, 217, 255, 255), 7: (255, 255, 255, 255), 8: (179, 174, 163, 255), 9: (0, 0, 0, 255), 10: (171, 0, 214, 255), 11: (0, 0, 0, 255), 12: (171, 0, 214, 255), 13: (171, 0, 214, 255), 14: (171, 0, 214, 255), 15: (171, 0, 214, 255), 16: (171, 0, 214, 255), 17: (171, 0, 214, 255), 18: (171, 0, 214, 255), 19: (171, 0, 214, 255), 20: (171, 0, 214, 255), 21: (171, 0, 214, 255), 22: (0, 0, 0, 255), 23: (171, 0, 214, 255), 24: (171, 0, 214, 255), 25: (171, 0, 214, 255), 26: (171, 0, 214, 255), 27: (171, 0, 214, 255), 28: (171, 0, 214, 255), 29: (171, 0, 214, 255), 30: (171, 0, 214, 255), 31: (171, 0, 214, 255), 32: (171, 0, 214, 255), 33: (0, 0, 0, 255), 34: (171, 0, 214, 255), 35: (171, 0, 214, 255), 36: (171, 0, 214, 255), 37: (171, 0, 214, 255), 38: (171, 0, 214, 255), 39: (171, 0, 214, 255), 40: (171, 0, 214, 255), 41: (171, 0, 214, 255),

In [6]:
# check that the tiles are aligned 
for p in [naip_file1, lcmap_file1]:
    with rasterio.open(p) as src:
        print(p, src.crs, src.res, src.shape, src.bounds, src.nodata, src.count, src.dtypes)

C:/Users/emna9376/OneDrive - UCB-O365/CIRES_IRP/naip_lcmap_pairs/hayman_co_m_3910554_se_13_060_20190913_20191121_image.tif EPSG:26913 (0.6, 0.6) (12260, 9700) BoundingBox(left=472788.0, bottom=4330476.0, right=478608.0, top=4337832.0) None 4 ('uint8', 'uint8', 'uint8', 'uint8')
C:/Users/emna9376/OneDrive - UCB-O365/CIRES_IRP/naip_lcmap_pairs/hayman_lcmap.tif EPSG:26913 (0.6, 0.6) (12260, 9700) BoundingBox(left=472788.0, bottom=4330476.0, right=478608.0, top=4337832.0) 255.0 1 ('uint8',)


In [7]:
# Copied from https://www.usgs.gov/special-topics/lcmap/lcmap-class-legend-and-description
LCMAP_CLASS_DEFINITIONS = {
    0: ("No Data", "No data value"),
    1: ("Developed", "Built-up land such as buildings, roads, and other constructed surfaces."),
    2: ("Cropland", "Land used to produce food, fiber, and fuel, whether vegetated or bare at the time."),
    3: ("Grass/Shrub", "Land dominated by grasses, forbs, or shrubs: at least 10% grass/shrub cover and under 10% tree cover."),
    4: ("Tree Cover", "Tree-covered land with tree cover density above 10%."),
    5: ("Water", "Streams, canals, lakes, reservoirs, bays, or ocean."),
    6: ("Wetland", "Land where water saturation controls soil, vegetation, and animal communities; a mosaic of water, bare soil, and herbaceous or woody cover."),
    7: ("Ice/Snow", "Perennial ice and snow that does not fully melt in summer."),
    8: ("Barren", "Natural soil, sand, or rock with less than 10% vegetation."),
}

# Copied from the embedded color table in the lCMAP files (above)
LCMAP_CLASS_COLORS = {
    0: (0, 0, 0, 0),
    1: (255, 50, 50, 255),     # Developed
    2: (190, 140, 90, 255),    # Cropland
    3: (230, 240, 210, 255),   # Grass/Shrub
    4: (28, 99, 48, 255),      # Tree Cover
    5: (0, 112, 255, 255),     # Water
    6: (179, 217, 255, 255),   # Wetland
    7: (255, 255, 255, 255),   # Ice/Snow
    8: (179, 174, 163, 255),   # Barren
}


In [14]:
########### STEP 2
# hand-coded target distribution
lcmap_codes = [1, 2, 3, 4, 5, 6, 7, 8]
lcmap_names = ["Developed", "Cropland", "Grass/Shrub", "Tree Cover",
               "Water", "Wetland", "Ice/Snow", "Barren"]

targets = ["Developed", "Cropland", "Grass/Shrub", "Tree Cover",
               "Water", "Wetland", "Ice/Snow", "Barren"]  

# rows = targets, cols = LCMAP classes
## first pass just use basic for the same classes where it is 90% the same class but then some fuzziness for the 10%
recipes = {
    #                 develop  crop  grass  tree  water  wetland  ice  barren
    "Developed":      [0.90,  0.05,  0.00,  0.00,  0.00,  0.00,  0.00,  0.05],
    "Cropland":       [0.00,  0.90,  0.05,  0.05,  0.00,  0.00,  0.00,  0.00],
    "Grass/Shrub":    [0.00,  0.05,  0.90,  0.05,  0.00,  0.00,  0.00,  0.00],
    "Tree Cover":     [0.00,  0.05,  0.50,  0.90,  0.00,  0.00,  0.00,  0.00],
    "Water":          [0.00,  0.00,  0.00,  0.00,  0.90,  0.05,  0.00,  0.05],
    "Wetland":        [0.00,  0.00,  0.05,  0.00,  0.05,  0.90,  0.00,  0.00],
    "Ice/Snow":       [0.00,  0.05,  0.00,  0.00,  0.00,  0.00,  0.90,  0.05],
    "Barren":         [0.00,  0.05,  0.05,  0.00,  0.00,  0.00,  0.00,  0.90],
}

M = pd.DataFrame(recipes, index=targets)[lcmap_names]   
M.columns = lcmap_codes                                


In [ ]:
M.to_csv("lcmap_to_targets.csv")                  
np.save("lcmap_to_targets.npy", M.to_numpy())   


In [19]:
# helper functions - adjusted from make_priors_chesapeake.py

def create_map_raw_lc_to_idx(class_definitions):
    LC_TO_IDX = np.zeros(np.array(list(class_definitions.keys())).max() + 1, dtype=np.uint8)
    for i, k in enumerate(class_definitions.keys()):
        LC_TO_IDX[k] = i
    return LC_TO_IDX  

def get_colors(class_colors):
    return np.array([class_colors[c] for c in class_colors.keys()]) / 255.0

def compile_and_save_one_hot_lcmap_cc(lcmap_fp, remap_to_idx=True):
    # this creates lookup table for lcmap codes and positions 
    if remap_to_idx: 
        lcmap = create_map_raw_lc_to_idx(LCMAP_CLASS_DEFINITIONS)[rasterio.open(lcmap_fp).read()[0]]
    else:
        lcmap = rasterio.open(lcmap_fp).read()[0]
        
    num_classes = len(get_colors(LCMAP_CLASS_COLORS))
    lcmap_onehot = np.zeros((num_classes, lcmap.shape[0],lcmap.shape[1]))
    for c in range(num_classes):
        lcmap_onehot[c][lcmap == c] = 1

    prof_out = rasterio.open(lcmap_fp).profile.copy()
    prof_out['count'] = num_classes
    
    lcmap_fp_out = lcmap_fp.replace('lcmap.', 'lcmap_onehot.')
    
    with rasterio.open(lcmap_fp_out, 'w', **prof_out) as f:
        f.write(lcmap_onehot)


# def compile_and_save_one_blurred_hot_lcmap_cc(lcmap_onehot_fp,
#                                           blur_kernel_size=101,
#                                           blur_sigma=51,
#                                           device=T.device('cuda:0')):
    
#     lcmap_onehot = rasterio.open(lcmap_onehot_fp).read()    
#     lcmap_onehot = T.tensor(lcmap_onehot, device=device).float()
    
#     # guassian blur
#     blur = GaussianBlur(blur_kernel_size, blur_sigma)
#     lcmap_onehot_blurred = blur(lcmap_onehot).cpu().numpy()
    

#     prof_out = rasterio.open(lcmap_onehot_fp).profile.copy()
    
#     lcmap_fp_out = lcmap_onehot_fp.replace('lcmap_onehot', 
#                                          f'lcmap_onehot_blurred_kernelsize_{blur_kernel_size}_sigma_{blur_sigma}')
#     with rasterio.open(lcmap_fp_out, 'w', **prof_out) as f:
#         f.write((lcmap_onehot_blurred * 255.).astype(np.uint8))


def compile_and_save_one_blurred_hot_lcmap_cc(lcmap_onehot_fp,
                                              blur_kernel_size=101,
                                              blur_sigma=51,
                                              device=T.device('cuda:0')):
    
    blur = GaussianBlur(blur_kernel_size, blur_sigma)

    with rasterio.open(lcmap_onehot_fp) as src:
        prof_out = src.profile.copy()
        data = src.read()                      # (C, H, W)

    out = np.empty(data.shape, dtype=np.uint8)
    with T.no_grad():
        for c in range(data.shape[0]):
            ch = T.from_numpy(data[c:c+1]).to(device).float()
            ch = blur(ch).clamp_(0, 1)
            out[c] = (ch * 255.).byte().cpu().numpy()[0]
            del ch
    T.cuda.empty_cache()

    lcmap_fp_out = lcmap_onehot_fp.replace(
        'lcmap_onehot',
        f'lcmap_onehot_blurred_kernelsize_{blur_kernel_size}_sigma_{blur_sigma}')
    prof_out.update(dtype='uint8')
    with rasterio.open(lcmap_fp_out, 'w', **prof_out) as f:
        f.write((out).astype(np.uint8))


def make_and_save_priors_cc(tiles, 
                            dir_this,
                            version, 
                            cooc_matrix,
                            plot_mapping_matrix=False):

    # by default use OSM
    ignore_osm = False
    ignore_buildings = False
    spatial_smooth = True
    
    #these index values are what they are for LCMAP
    road_idx_cc = 1
    building_idx_cc = 1
    water_idx_cc = 5
        
    if version == 'from_cooccurrences_101_31_no_osm_no_buildings':
        lcmap_blur_kernelsize = 101
        lcmap_blur_sigma = 31
        ignore_osm = True
        ignore_buildings = True
            
    else:
        print(f'version {version} not understood')
        return
    
    print(version)
    if spatial_smooth: print(lcmap_blur_sigma)
    else: print("not performing spatial smooth")
        
        
    if plot_mapping_matrix:
        plt.figure()
        plt.imshow(nlcd_mapping_matrix)
        plt.title('mapping matrix being used');
        plt.show()
        
    for i,tile_id in enumerate(tiles):
        print(i, tile_id)
        t1 = time.time()

        if spatial_smooth:
            # note this assumes that you've made this blurred input already.
            lcmap_fp = f'{dir_this}/{tile_id}_lcmap_onehot_blurred_kernelsize_' + \
                            f'{lcmap_blur_kernelsize}_sigma_{lcmap_blur_sigma}.tif'
            
        else:
            lcmap_fp = f'{dir_this}/{tile_id}_lcmap_onehot.tif'
        print(lcmap_fp)
        
        #by default these will get ignored
        roads_fp = f'{dir_this}/m_{tile_id}_roads.tif'
        buildings_fp = f'{dir_this}/m_{tile_id}_buildings.tif'
        waterways_fp = f'{dir_this}/m_{tile_id}_waterways.tif'
        waterbodies_fp = f'{dir_this}/m_{tile_id}_waterbodies.tif'
        
        if not ignore_buildings:
            buildings = rasterio.open(buildings_fp).read()[0]
            
        if not ignore_osm:
            roads = rasterio.open(roads_fp).read()[0]
            waterways = rasterio.open(waterways_fp).read()[0]
            waterbodies = rasterio.open(waterbodies_fp).read()[0]
            water = np.maximum(waterways, waterbodies)
            water = np.maximum(waterbodies, util.blur(waterways,waterways_preblur_radius)==1)

        # blur water
        if not ignore_osm:
            roads = util.bag_from_numpy(roads.astype('float'),radius=roads_bag_radius)
            water = util.bag_from_numpy(water.astype('float'),radius=water_bag_radius)
            
        if not ignore_buildings:
            buildings = util.bag_from_numpy(buildings.astype('float'),radius=buildings_bag_radius)

        with rasterio.open(lcmap_fp) as lcmap_preblurred:
            if ignore_buildings:
                print('ignoring buildings')
                buildings = np.zeros(lcmap_preblurred.shape[1:])
            
            if ignore_osm:
                print('ignoring roads, and water')
                roads = np.zeros(buildings.shape)
                water = np.zeros(buildings.shape)
                
            # using the same script from previous - general_prior_calculation.py
            prof_out = lcmap_preblurred.profile.copy()
            prior_this_5 = calculate_prior_preblurred(lcmap_preblurred.read(),
                                                roads,
                                                buildings,
                                                water,
                                                cooc_matrix,
                                                road_idx = road_idx_cc,
                                                building_idx = building_idx_cc,
                                                water_idx = water_idx_cc)
        
        # now exclude nodata classes from the prior
        prior_this = prior_this_5[1:]
        
        # write this prior
        prof_out['count'] = prior_this.shape[0]
        with rasterio.open(f'{dir_this}/{tile_id}_prior_{version}.tif', 'w', **prof_out) as f:
            f.write((prior_this*255.).astype(np.uint8))

        t2 = time.time()
        print(t2-t1, ' seconds')
 

In [11]:
######## STEP 3
# create the prior: one-hot, blur, use target matrix

#### INSTEAD OF TILE ID - USE FIRE NAME!
 
# 1. save the onehot NLCD and blurred onehot_nlcd
lcmap_fp = f'{pairs_path}{file1_firename}_lcmap.tif'

In [ ]:
# save onehot nlcd
compile_and_save_one_hot_lcmap_cc(lcmap_fp) 

In [12]:
# rename file
lcmap_fp_onehot = lcmap_fp.replace('lcmap.','lcmap_onehot.')

In [ ]:
# save blurred nlcd
compile_and_save_one_blurred_hot_lcmap_cc(lcmap_fp_onehot,
                                         blur_kernel_size=101,
                                         blur_sigma=31,
                                         device=device) 


In [15]:
# load hand coded matrix
cooc_matrix_file1 = M
# when file is saved: 
#cooc_matrix_file1 = np.load('lcmap_to_targets.npy') # to be changed to be more modular when needed


In [ ]:
# 2. make the priors
# where to find this data
tiles = [file1_firename]

make_and_save_priors_cc(tiles,  
                        pairs_path, 
                        'from_cooccurrences_101_31_no_osm_no_buildings',  
                        cooc_matrix_file1,
                        plot_mapping_matrix=False)

from_cooccurrences_101_31_no_osm_no_buildings
31
0 hayman
/data-store/iplant/home/emilynagamoto/NAIP_LCMAP//hayman_lcmap_onehot_blurred_kernelsize_101_sigma_31.tif
ignoring buildings
ignoring roads, and water


In [ ]:
# set something up in weights and bias where i monitor the model training that 
# shows me how the output of predictive model changes over time
# finnicky to train
# will get weird degenerate responses
# trudge through and plot things, if loss is going down but output looks bad